In [110]:
import pandas as pd
import numpy as np
import ast
from sentence_transformers import SentenceTransformer

In [111]:
df_rez = pd.read_csv('df_rez_new.csv')[['id', 'resume_title', 'text_clean', 'skills_list', 'experience_text']]
df_vac = pd.read_csv('df_vac_new.csv')[['id', 'vacancy_name', 'text_clean', 'skills_final', 'experience_years_min']]

In [112]:
def restore_list(value):
    return ast.literal_eval(value)

df_rez['skills_list'] = df_rez['skills_list'].apply(restore_list)
df_vac['skills_final'] = df_vac['skills_final'].apply(restore_list)

Перенести в EDA

In [ ]:
# (df_vac['skills_final'].apply(lambda x: len(x) == 0)).sum()

np.int64(22)

In [ ]:
# df_vac = df_vac[df_vac['skills_final'].apply(lambda x: len(x) > 0)].copy()

## Псевдоэталон

In [117]:
def calc_exp_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0.0
    if pd.isna(vacancy_exp):
        vacancy_exp = 0.0

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0
    
    return resume_exp / vacancy_exp

In [118]:
def calc_skill_score(resume_skills, vacancy_skills):
    matched_skills = len(set(resume_skills) & set(vacancy_skills))
    skill_score = matched_skills / len(vacancy_skills)
    return skill_score, matched_skills

In [119]:
df_pairs = df_rez.merge(df_vac, how='cross')

In [120]:
df_pairs[['skill_score', 'matched_skills']] = df_pairs.apply(
    lambda row: pd.Series(
        calc_skill_score(row['skills_list'], row['skills_final'])
    ),
    axis=1
)

In [121]:
df_pairs['exp_score'] = df_pairs.apply(
    lambda row: calc_exp_score(row['experience_text'], row['experience_years_min']),
    axis=1
)

In [122]:
df_pairs['final_score'] = 0.67 * df_pairs['skill_score'] + 0.33 * df_pairs['exp_score']

In [123]:
df_pairs = df_pairs.sort_values(
    by=['id_x', 'final_score', 'matched_skills', 'exp_score'],
    ascending=[True, False, False, False]
).copy()

In [124]:
df_pairs['rank_pseudo'] = df_pairs.groupby('id_x').cumcount() + 1

In [125]:
pseudo_reference = df_pairs[[
    'id_x',
    'resume_title',
    'id_y',
    'vacancy_name',
    'skill_score',
    'matched_skills',
    'exp_score',
    'final_score',
    'rank_pseudo'
]].copy()

In [126]:
pseudo_reference = pseudo_reference.rename(columns={
    'id_x': 'resume_id',
    'id_y': 'vacancy_id'
})

## Построение baseline

In [127]:
df_rez.head()

,id,resume_title,text_clean,skills_list,experience_text
0,2,Аналитик данных,сентябрь 2024 сентябрь 2025 кофемания москва г...,"[пользователь пк, ms sql, ms office, driving l...",8.92
1,3,Аналитик данных,апрель 2025 по настоящее время termoland услуг...,"[ms powerpoint, python, numpy, sql, git, pycha...",6.50
2,4,Аналитик данных,декабрь 2024 по настоящее время центральный ба...,"[data science, регрессионный анализ, проверка ...",2.67
3,6,Аналитик данных,январь 2025 по настоящее время нева дельта спб...,"[sql, power bi, ms power bi, dax, tableau, abc...",NaN
4,8,Аналитик данных,октябрь 2011 по настоящее время сбер москва ra...,"[python, vba, sql, ms office, oracle, qlik sen...",14.42


In [156]:
N = 10  # сколько примеров сохранить

with open("text_clean_samples.txt", "w", encoding="utf-8") as f:
    for i, text in enumerate(df_rez["text_clean"].head(N), 1):
        f.write(f"--- Пример {i} ---\n")
        f.write(str(text) + "\n\n")
        

In [128]:
df_vac.head()

,id,vacancy_name,text_clean,skills_final,experience_years_min
0,15,Аналитик данных / экономист инвестиционных про...,46 органов исполнительной власти 132 территори...,"[python, django, pandas, ms office (power poin...",1.0
1,34,Data Engineer (разработчик DWH),задаем тренды в технологиях ритейла x5 group —...,"[sql, python, big data, apache airflow, apache...",3.0
2,92,Аналитик данных,центр компетенций бизнес аналитики и финансово...,"[sql, python, hadoop, excel]",1.0
3,162,Data Engineer по построению DWH,компания ecofinance развивает и внедряет проду...,"[etl, sql, dwh, apache kafka, debezium, bi, db...",3.0
4,65,Senior Data Scientist,создавай инновации в атмосфере свободы — и рас...,"[python, ml-библиотеки, sql]",1.0


In [129]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2276.85it/s]


In [130]:
resume_texts = df_rez['text_clean'].tolist()
vacancy_texts = df_vac['text_clean'].tolist()

In [131]:
resume_embeddings = model.encode(
    resume_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

vacancy_embeddings = model.encode(
    vacancy_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 46/46 [00:26<00:00,  1.72it/s]


In [132]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(resume_embeddings, vacancy_embeddings)
sim_matrix.shape

(11940, 1443)

In [133]:
def get_top_matches(sim_matrix, df_rez_base, df_vac_base, top_k=5):
    results = []

    for i in range(sim_matrix.shape[0]):
        scores = sim_matrix[i]
        top_idx = np.argsort(scores)[::-1][:top_k]

        for rank, j in enumerate(top_idx, start=1):
            results.append({
                'resume_id': df_rez_base.iloc[i]['id'],
                'resume_title': df_rez_base.iloc[i]['resume_title'],
                'vacancy_id': df_vac_base.iloc[j]['id'],
                'vacancy_name': df_vac_base.iloc[j]['vacancy_name'],
                'rank': rank,
                'semantic_score': scores[j]
            })

    return pd.DataFrame(results)

In [134]:
top_matches = get_top_matches(sim_matrix, df_rez, df_vac, top_k=5)
top_matches.head(10)

,resume_id,resume_title,vacancy_id,vacancy_name,rank,semantic_score
0,2,Аналитик данных,10557,BI-аналитик,1,0.653296
1,2,Аналитик данных,12318,Data Analyst,2,0.628879
2,2,Аналитик данных,13671,Аналитик данных - статистик,3,0.628866
3,2,Аналитик данных,16748,Аналитик данных/статистик,4,0.626059
4,2,Аналитик данных,12712,Аналитик данных (эксперт Excel),5,0.609621
5,3,Аналитик данных,15015,Data Analyst (Middle),1,0.679967
6,3,Аналитик данных,11006,Senior Data Engineer,2,0.678050
7,3,Аналитик данных,12197,Senior Data Scientist (ЦУНДО),3,0.673552
8,3,Аналитик данных,15243,Senior Data Scientist (ЦУНДО),4,0.673552
9,3,Аналитик данных,10450,Аналитик данных,5,0.664285


## Оценка качества

In [164]:
def calc_skill_score(resume_skills, vacancy_skills):
    if not isinstance(resume_skills, list):
        resume_skills = []

    if not isinstance(vacancy_skills, list):
        vacancy_skills = []

    resume_skills = set([str(x).lower().strip() for x in resume_skills])
    vacancy_skills = set([str(x).lower().strip() for x in vacancy_skills])

    if len(vacancy_skills) == 0:
        return 0.0, 0

    matched_skills = len(resume_skills & vacancy_skills)
    skill_score = matched_skills / len(vacancy_skills)

    return skill_score, matched_skills


def calc_exp_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0

    if pd.isna(vacancy_exp):
        vacancy_exp = 0

    resume_exp = float(resume_exp)
    vacancy_exp = float(vacancy_exp)

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0

    return resume_exp / vacancy_exp

In [165]:
top_matches_eval = (
    top_matches
    .merge(
        df_rez[["id", "resume_title", "skills_list", "experience_text"]],
        left_on="resume_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
    .merge(
        df_vac[["id", "vacancy_name", "skills_final", "experience_years_min"]],
        left_on="vacancy_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
)

top_matches_eval.head()

,resume_id,resume_title_x,vacancy_id,vacancy_name_x,rank,semantic_score,resume_title_y,skills_list,experience_text,vacancy_name_y,skills_final,experience_years_min
0,2,Аналитик данных,10557,BI-аналитик,1,0.653296,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,BI-аналитик,"[sql, datalens, clickhouse, sucd, python, etl]",1.0
1,2,Аналитик данных,12318,Data Analyst,2,0.628879,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Data Analyst,"[английский — b1 — средний, python, sql, опыт ...",1.0
2,2,Аналитик данных,13671,Аналитик данных - статистик,3,0.628866,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных - статистик,"[анализ данных, статистика, сэд, ms office, ms...",NaN
3,2,Аналитик данных,16748,Аналитик данных/статистик,4,0.626059,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных/статистик,"[работа с большим объемом информации, аналитич...",NaN
4,2,Аналитик данных,12712,Аналитик данных (эксперт Excel),5,0.609621,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных (эксперт Excel),"[ms excel, vba/macros]",1.0


In [166]:
top_matches_eval[["skill_score", "matched_skills"]] = top_matches_eval.apply(
    lambda row: pd.Series(
        calc_skill_score(
            row["skills_list"],
            row["skills_final"]
        )
    ),
    axis=1
)

top_matches_eval["exp_score"] = top_matches_eval.apply(
    lambda row: calc_exp_score(
        row["experience_text"],
        row["experience_years_min"]
    ),
    axis=1
)

In [167]:
top_matches_eval[["skill_score", "matched_skills"]] = top_matches_eval.apply(
    lambda row: pd.Series(
        calc_skill_score(row["skills_list"], row["skills_final"])
    ),
    axis=1
)

top_matches_eval["exp_score"] = top_matches_eval.apply(
    lambda row: calc_exp_score(
        row["experience_text"],
        row["experience_years_min"]
    ),
    axis=1
)

In [168]:
top_matches_eval[[
    "resume_id",
    "vacancy_id",
    "rank",
    "semantic_score",
    "skill_score",
    "matched_skills",
    "exp_score"
]].head()

,resume_id,vacancy_id,rank,semantic_score,skill_score,matched_skills,exp_score
0,2,10557,1,0.653296,0.166667,1.0,1.0
1,2,12318,2,0.628879,0.200000,1.0,1.0
2,2,13671,3,0.628866,0.142857,2.0,1.0
3,2,16748,4,0.626059,0.086957,2.0,1.0
4,2,12712,5,0.609621,0.000000,0.0,1.0


In [175]:
# Среднее совпадение навыков в top-5
mean_skill_score_5 = (
    top_matches_eval[top_matches_eval["rank"] <= 5]["skill_score"]
    .mean()
)


# Good Match Rate@5
def good_match_rate_at_k(df, k=5, skill_threshold=0.3, exp_threshold=0.7):
    temp = df[df["rank"] <= k].copy()

    temp["is_good"] = (
        (temp["skill_score"] >= skill_threshold) &
        (temp["exp_score"] >= exp_threshold)
    )

    return temp.groupby("resume_id")["is_good"].max().mean()


good_match_rate_5 = good_match_rate_at_k(
    top_matches_eval,
    k=5,
    skill_threshold=0.3,
    exp_threshold=0.7
)


# Experience Fit Rate@5
def experience_fit_rate_at_k(df, k=5, exp_threshold=0.7):
    temp = df[df["rank"] <= k].copy()

    temp["exp_fit"] = temp["exp_score"] >= exp_threshold

    return temp.groupby("resume_id")["exp_fit"].mean().mean()


experience_fit_rate_5 = experience_fit_rate_at_k(
    top_matches_eval,
    k=5,
    exp_threshold=0.7
)


# Итоговая таблица
metrics_summary = pd.DataFrame({
    "metric": [
        "Good Match Rate@5",
        "Experience Fit Rate@5",
        "Mean skill_score@5"
    ],
    "value": [
        good_match_rate_5,
        experience_fit_rate_5,
        mean_skill_score_5
    ]
})

metrics_summary

,metric,value
0,Good Match Rate@5,0.530737
1,Experience Fit Rate@5,0.776516
2,Mean skill_score@5,0.216425
